In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from selenium.webdriver.chrome.options import Options
import time
import os
import pandas as pd

In [2]:
link_df = pd.read_csv("medicine_links.csv")
link_df.head()

,category,link
0,"Thuốc kháng sinh, Kháng nấm",https://nhathuocminhchau.com.vn/apilevo-500-ap...
1,"Thuốc kháng sinh, Kháng nấm",https://zalo.me/840415224445769890
2,"Thuốc kháng sinh, Kháng nấm",https://nhathuocminhchau.com.vn/vigentin-87512...
3,"Thuốc kháng sinh, Kháng nấm",https://nhathuocminhchau.com.vn/novofenti-200m...
4,"Thuốc kháng sinh, Kháng nấm",https://nhathuocminhchau.com.vn/cotrimoxazol-4...


In [3]:
print(f"Số link trùng lặp: {link_df.duplicated(subset=['link']).sum()}")
print(f"Số link duy nhất: {link_df['link'].nunique()}")

Số link trùng lặp: 55658
Số link duy nhất: 9545


In [4]:
print("Danh sách các link trùng lặp:")
print(link_df[link_df.duplicated(subset=['link'], keep=False)]['link'].value_counts())
for link, count in link_df[link_df.duplicated(subset=['link'], keep=False)]['link'].value_counts().items():
    print(f"{link}: {count} lần")

Danh sách các link trùng lặp:
link
https://zalo.me/840415224445769890                                                               33
https://nhathuocminhchau.com.vn/thuoc-jasugrel-film-coated-tablets-10mg                          32
https://nhathuocminhchau.com.vn/thuoc-dozidine-mr-35mg-trimetazidine-35mg-hop-60-vien            32
https://nhathuocminhchau.com.vn/thuoc-tim-mach-rofba-30-vien                                     32
https://nhathuocminhchau.com.vn/thuoc-tim-mach-cadila-pharm-aldarone-hop-100-vien                32
                                                                                                 ..
https://nhathuocminhchau.com.vn/dich-truyen-tienam-500                                            2
https://nhathuocminhchau.com.vn/thuoc-nho-mat-cravit-15-levofloxacin-hydrat-15mg-hop-1-lo-5ml     2
https://nhathuocminhchau.com.vn/thuoc-lomexin-600mg-hop-2-vien                                    2
https://nhathuocminhchau.com.vn/bactronil                        

In [5]:
# Các link trùng category
print("Danh sách các link trùng category:")
print(link_df[link_df.duplicated(subset=['category'], keep=False)]['category'].value_counts())
for category, count in link_df[link_df.duplicated(subset=['category'], keep
=False)]['category'].value_counts().items():
    print(f"{category}: {count} lần")

Danh sách các link trùng category:
category
Thuốc da liễu                           2336
Thuốc gout, cơ, xương khớp              2331
Thuốc trị viêm gan B,C & HIV            2330
Thuốc kháng viêm, giảm đau & hạ sốt     2320
Thuốc Tiêu Hóa, gan mật                 2309
Thuốc tiêm, dịch truyền                 2259
Thuốc tác dụng đối với máu              2202
Thuốc phụ khoa                          2198
Thuốc chống thải ghép                   2185
Thuốc gây mê, gây tê                    2185
Thuốc điều hòa miễn dịch                2170
Thuốc mọc tóc                           2165
Thuốc chống say xe                      2162
Thuốc trĩ                               2156
Thuốc ngừa thai                         2151
Thuốc kiểm soát đặc biệt                2151
Thuốc trị ung thư, u bướu               2143
Thuốc cấp cứu và giải độc               2132
Thuốc cầm máu                           2118
Thuốc bổ - Vitamin - Khoáng chất        2103
Thuốc dùng ngoài                        2046
Thuốc hướng

In [6]:
# Loại bỏ trùng lặp
link_df = link_df.drop_duplicates(subset=['link'])
print(f"Số link sau khi loại bỏ trùng lặp: {len(link_df)}")

Số link sau khi loại bỏ trùng lặp: 9545


In [7]:
# Số link còn lại ở mỗi category
print("Số link còn lại ở mỗi category:")
print(link_df['category'].value_counts())

Số link còn lại ở mỗi category:
category
Thuốc kháng sinh, Kháng nấm             1113
Thuốc tim mạch & Huyết áp               1090
Thuốc hướng thần & Cai nghiện            919
Thuốc dùng ngoài                         902
Thuốc kháng viêm, giảm đau & hạ sốt      857
Thuốc Tiêu Hóa, gan mật                  811
Thuốc Hô Hấp                             556
Thuốc trị ung thư, u bướu                489
Thuốc bổ - Vitamin - Khoáng chất         433
Thuốc tiêm, dịch truyền                  343
Thuốc chống dị ứng ( kháng histamin)     329
Thuốc tiểu đường                         298
Thuốc kháng Virus                        295
Thuốc gout, cơ, xương khớp               287
Thuốc Hocmon, Nội tiết tố                163
Thuốc cường dương                        124
Thuốc đường tiết niệu                     85
Thuốc da liễu                             49
Thuốc tác dụng đối với máu                47
Thuốc ngừa thai                           47
Thuốc phụ khoa                            44
Thuốc giãn mạc

In [8]:
# Loại bỏ các link không phải là thuốc (không có từ thuốc trong link)
link_df = link_df[link_df['link'].str.contains('thuoc')]
print(f"Số link sau khi loại bỏ các link không phải là thuốc: {len(link_df)}")
# Số link còn lại ở mỗi category sau khi loại bỏ các link không phải là thuốc
print("Số link còn lại ở mỗi category sau khi loại bỏ các link không phải là thuốc:")
print(link_df['category'].value_counts())


Số link sau khi loại bỏ các link không phải là thuốc: 9544
Số link còn lại ở mỗi category sau khi loại bỏ các link không phải là thuốc:
category
Thuốc kháng sinh, Kháng nấm             1112
Thuốc tim mạch & Huyết áp               1090
Thuốc hướng thần & Cai nghiện            919
Thuốc dùng ngoài                         902
Thuốc kháng viêm, giảm đau & hạ sốt      857
Thuốc Tiêu Hóa, gan mật                  811
Thuốc Hô Hấp                             556
Thuốc trị ung thư, u bướu                489
Thuốc bổ - Vitamin - Khoáng chất         433
Thuốc tiêm, dịch truyền                  343
Thuốc chống dị ứng ( kháng histamin)     329
Thuốc tiểu đường                         298
Thuốc kháng Virus                        295
Thuốc gout, cơ, xương khớp               287
Thuốc Hocmon, Nội tiết tố                163
Thuốc cường dương                        124
Thuốc đường tiết niệu                     85
Thuốc da liễu                             49
Thuốc tác dụng đối với máu                47


In [9]:
# Lưu file đã loại bỏ trùng lặp
link_df.to_csv("medicine_links_deduplicated.csv", index=False)